# Understanding occupational mobility and automation
Run cells from top to bottom (Shift+Enter). This notebook uses the original Python approximation. The three-occupation example is illustrative, not a reproduction of a paper figure.

The Julia agent simulation implements individual random events; here we calculate expected worker flows.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "code" / "labornet.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "code" / "labornet.py").exists(), "Start Jupyter from the project folder"
sys.path.insert(0, str(ROOT / "code"))
import labornet as lbn

## 1. Three occupations
Rows of A are source occupations; columns are destinations. Larger weights mean easier transitions. Workers choose destinations in proportion to A[i,j] times vacancies[j]. Unemployment is indexed by the previous occupation.

In [ ]:
names = ["Occupation A", "Occupation B", "Occupation C"]
A = np.array([[1., .3, .05], [.3, 1., .3], [.05, .3, 1.]])
e0 = np.array([1000., 1000., 1000.])
u0 = np.array([50., 50., 50.])
v0 = np.array([30., 30., 30.])
pd.DataFrame(A, index=names, columns=names)

## 2. Applications and hiring
Q gives application probabilities. Expected applications are unemployed workers times Q. The hiring probability accounts for competition for vacancies.

In [ ]:
Q = A * v0
Q = Q / Q.sum(axis=1, keepdims=True)
applications = u0[:, None] * Q
hiring_probability = lbn.matching_probability(applications, v0)
flows = applications * hiring_probability
display(pd.DataFrame(Q, index=names, columns=names))
display(pd.DataFrame(flows, index=names, columns=names))

## 3. Run a demand shock
We use the original fire_and_hire_workers function. At step 100, target demand moves from occupation A to C while total target demand stays constant. Compare with a baseline without a shock. Rates below are illustrative. Each run starts with fresh arrays because the original function updates some inputs in place.

In [ ]:
def simulate(shocked=True, steps=300):
    e, u, v = e0.copy(), u0.copy(), v0.copy()
    history = []
    for t in range(steps):
        target = e0 + v0
        if shocked and t >= 100:
            target = target + np.array([-300., 0., 300.])
        u, v, e, _, _ = lbn.fire_and_hire_workers(
            0.016, 0.012, 0.16, 0.16, e, u, v, target, A, lbn.matching_probability)
        assert np.all(np.isfinite(u)) and np.all(u >= 0)
        assert np.isclose((e + u).sum(), (e0 + u0).sum())
        history.append([*e, *u, *v])
    return np.array(history)

baseline = simulate(False)
shock = simulate(True)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for result, label in [(baseline, "Baseline"), (shock, "Demand shock")]:
    axes[0].plot(100 * result[:, 3:6].sum(axis=1) / (e0 + u0).sum(), label=label)
axes[0].set(xlabel="Model step", ylabel="Unemployment (%)")
axes[0].legend()
for i, name in enumerate(names):
    axes[1].plot(shock[:, i], label=name)
axes[1].set(xlabel="Model step", ylabel="Employment")
axes[1].legend()
for ax in axes:
    ax.axvline(100, color="grey", linestyle="--")
plt.tight_layout()
plt.show()

## 4. Inspect the paper data
These cells load the real network and occupation characteristics without starting the original long-running scripts. Inspect the columns before selecting variables.

In [ ]:
network = np.loadtxt(ROOT / "data" / "occupational_mobility_network.csv", delimiter=",")
occupations = pd.read_csv(ROOT / "data" / "ipums_variables.csv")
employment = pd.read_csv(ROOT / "data" / "ipums_employment_2016.csv")
print("Network shape:", network.shape)
print("Occupation columns:", occupations.columns.tolist())
display(occupations.head())
display(employment.head())

## Things to try
- Reduce the A-to-C network weight and rerun from section 1.
- Change the size of the demand shift in simulate.
- Compare the unemployment peak and the time it takes to adjust.

Next, read code/RunNumericalShock.py for the paper configuration, including its gradual shock, calibration and network variants. The simplified example above omits those details. The original scripts use relative paths and write result files; inspect their settings before running them.

## 5. Recreate Figure 1(b): the occupation network
Each node is an occupation; connections represent possible worker transitions.
Blue means low probability of computerization, red means high probability.
Node area is proportional to log employment (2010 employment from ipums_variables.csv).

This recreates the data encoding, not the exact published layout: original coordinates and plotting settings are not supplied. We use a seeded spring layout on symmetrized mobility weights. All positive off-diagonal directed links are retained for drawing; arrows are omitted for readability. Hover over a node for its occupation details; pan, zoom, and box-select nodes interactively.

The paper labels a node "statistical technicians"; this exact label is absent from the supplied metadata, so this recreation annotates only the three exact occupation-name matches.


In [ ]:
import networkx as nx
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

# Standalone: run this cell even if the earlier model cells have not been run.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
weights = np.loadtxt(ROOT / "data/occupational_mobility_network.csv", delimiter=",")
meta = pd.read_csv(ROOT / "data/ipums_variables.csv").sort_values("id").reset_index(drop=True)
assert weights.shape == (len(meta), len(meta))
assert np.array_equal(meta["id"].to_numpy(), np.arange(len(meta)))
probability = meta["auto_prob_average"].to_numpy(dtype=float)
employment = meta["total_employment_2010"].to_numpy(dtype=float)
assert np.isfinite(weights).all() and (weights >= 0).all()
assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
assert np.isfinite(employment).all() and (employment > 1).all()

# Remove self-loops for visualization; retain them in the original model data.
W = weights.copy()
np.fill_diagonal(W, 0)
G = nx.from_numpy_array((W + W.T) / 2)
positions = nx.spring_layout(G, seed=42, iterations=250, weight="weight")
xy = np.array([positions[i] for i in range(len(meta))])
source, destination = np.nonzero(W > 0)
segments = np.stack([xy[source], xy[destination]], axis=1)
cmap = plt.get_cmap("RdYlBu_r")
norm = Normalize(0, 1)

fig, ax = plt.subplots(figsize=(10, 11))
edge_colors = cmap(norm((probability[source] + probability[destination]) / 2))
ax.add_collection(LineCollection(segments, colors=edge_colors, linewidths=0.3, alpha=0.12, zorder=1))
node_sizes = 2.5 * np.log(employment)
nodes = ax.scatter(xy[:, 0], xy[:, 1], s=node_sizes, c=probability,
                   cmap=cmap, norm=norm, linewidths=0.15, edgecolors="white", zorder=2)
for label, offset in {"statistical technicians": (-75, 25), "pharmacy aides": (25, 30),
                      "childcare workers": (30, -30), "electricians": (-70, -30)}.items():
    matches = meta.index[meta["label"].str.casefold() == label].tolist()
    if matches:
        i = matches[0]
        ax.annotate(label, xy=xy[i], xytext=offset, textcoords="offset points",
                    fontsize=9, arrowprops={"arrowstyle": "-", "color": "0.25", "lw": 0.6}, zorder=3)
ax.set_title("Occupational mobility and automatability — Figure 1(b) recreation")
ax.set_aspect("equal")
ax.axis("off")
fig.colorbar(nodes, ax=ax, shrink=0.45, pad=0.01, label="Probability of computerization")
fig.tight_layout()
output = ROOT / "results" / "fig"
output.mkdir(parents=True, exist_ok=True)
fig.savefig(output / "figure_1b_recreation.png", dpi=220, bbox_inches="tight")
print(f"{len(meta)} occupations; {len(source):,} directed transitions (self-loops excluded)")
print("Saved:", output / "figure_1b_recreation.png")
plt.show()


In [ ]:
# Interactive version: hover, pan, zoom, and select occupations.
import networkx as nx
import plotly.graph_objects as go
from pathlib import Path
import numpy as np
import pandas as pd

# Standalone: run this cell even if the earlier model cells have not been run.
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
weights = np.loadtxt(ROOT / "data/occupational_mobility_network.csv", delimiter=",")
meta = pd.read_csv(ROOT / "data/ipums_variables.csv").sort_values("id").reset_index(drop=True)
assert weights.shape == (len(meta), len(meta))
assert np.array_equal(meta["id"].to_numpy(), np.arange(len(meta)))
probability = meta["auto_prob_average"].to_numpy(dtype=float)
employment = meta["total_employment_2010"].to_numpy(dtype=float)
assert np.isfinite(weights).all() and (weights >= 0).all()
assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
assert np.isfinite(employment).all() and (employment > 1).all()

# Preserve the static recreation's seeded layout and omit self-loops only for display.
W = weights.copy()
np.fill_diagonal(W, 0)
G = nx.from_numpy_array((W + W.T) / 2)
positions = nx.spring_layout(G, seed=42, iterations=250, weight="weight")
xy = np.array([positions[i] for i in range(len(meta))])
source, destination = np.nonzero(W > 0)

# Match the static plot: links inherit the average automation color of their endpoints.
# Group links into narrow color bands to keep the interactive chart responsive.
from plotly.colors import sample_colorscale
edge_probability = (probability[source] + probability[destination]) / 2
edge_bin = np.minimum((edge_probability * 20).astype(int), 19)
fig = go.Figure()
for band in range(20):
    selected = edge_bin == band
    if not selected.any():
        continue
    sx, tx = xy[source[selected], 0], xy[destination[selected], 0]
    sy, ty = xy[source[selected], 1], xy[destination[selected], 1]
    edge_x = np.column_stack([sx, tx, np.full(len(sx), np.nan)]).ravel()
    edge_y = np.column_stack([sy, ty, np.full(len(sy), np.nan)]).ravel()
    rgb = tuple(int(channel.strip()) for channel in sample_colorscale("RdYlBu_r", [((band + 0.5) / 20)])[0][4:-1].split(","))
    fig.add_trace(go.Scattergl(
        x=edge_x, y=edge_y, mode="lines",
        line=dict(color=f"rgba({rgb[0]},{rgb[1]},{rgb[2]},0.12)", width=0.4),
        hoverinfo="skip", name="Transitions", legendgroup="transitions", showlegend=(band == 0)))

# Degree and total outgoing mobility weight summarize each occupation's network links.
out_degree = (W > 0).sum(axis=1)
out_weight = W.sum(axis=1)
customdata = np.column_stack([meta["label"], employment, probability, out_degree, out_weight])
node_area = 2.5 * np.log(employment)
# Matplotlib scatter uses area; Plotly marker size uses diameter. Convert area to a
# visually comparable pixel diameter instead of reusing the area values directly.
node_sizes = 1.5 * np.sqrt(node_area)
fig.add_trace(go.Scattergl(
    x=xy[:, 0], y=xy[:, 1], mode="markers", name="Occupations",
    customdata=customdata,
    marker=dict(size=node_sizes, color=probability, cmin=0, cmax=1,
                colorscale="RdYlBu_r", reversescale=False,
                colorbar=dict(title="Probability of<br>computerization", tickformat=".0%"),
                line=dict(color="white", width=0.2), opacity=0.94),
    hovertemplate=("<b>%{customdata[0]}</b><br>Employment (2010): %{customdata[1]:,.0f}" +
                   "<br>Probability of computerization: %{customdata[2]:.1%}" +
                   "<br>Outgoing links: %{customdata[3]:,}" +
                   "<br>Total outgoing mobility weight: %{customdata[4]:.3f}<extra></extra>")))

# Retain the named annotations used in the static recreation when exact labels exist.
for label, offset in {"pharmacy aides": (25, 30), "childcare workers": (30, -30),
                      "electricians": (-70, -30)}.items():
    matches = meta.index[meta["label"].str.casefold() == label].tolist()
    if matches:
        i = matches[0]
        fig.add_annotation(x=xy[i, 0], y=xy[i, 1], text=label, showarrow=True,
                           ax=offset[0], ay=-offset[1], arrowhead=0,
                           font=dict(size=10, color="#263238"),
                           bgcolor="rgba(255,255,255,0.78)", borderpad=2)

fig.update_layout(
    title="Occupational mobility and automatability — Figure 1(b) recreation",
    template="plotly_white", width=950, height=950, paper_bgcolor="white", plot_bgcolor="white",
    hovermode="closest", dragmode="pan",
    xaxis=dict(visible=False, scaleanchor="y", scaleratio=1, showgrid=False, zeroline=False),
    yaxis=dict(visible=False, showgrid=False, zeroline=False),
    legend=dict(orientation="h", y=1.02, x=0, groupclick="togglegroup"),
    margin=dict(l=15, r=25, t=75, b=15),
    annotations=list(fig.layout.annotations) + [dict(
        text="Scroll to zoom · drag to pan · use box or lasso select to inspect groups",
        x=0, y=-0.035, xref="paper", yref="paper", showarrow=False,
        xanchor="left", font=dict(size=11, color="#667085")
    )])
output = ROOT / "results" / "interactive"
output.mkdir(parents=True, exist_ok=True)
html_path = output / "figure_1b_occupation_network.html"
fig.write_html(html_path, include_plotlyjs=True,
               config={"displaylogo": False, "scrollZoom": True, "responsive": True})
print(f"{len(meta)} occupations; {len(source):,} directed transitions (self-loops excluded)")
print("Saved standalone interactive chart:", html_path)
fig.show(config={"displaylogo": False, "scrollZoom": True, "responsive": True})


## 6. The 30 highest and 30 lowest automation probabilities
These charts rank occupations by `auto_prob_average`, the paper's mapped Frey–Osborne probability of computerization. They describe estimated technical automatability, not predicted job losses or the model's unemployment outcomes. Each panel has exactly 30 occupations; equal scores are ordered alphabetically to make selection reproducible. The two panels use different horizontal ranges so that low probabilities remain readable.

Data caveat: 17 occupations have an exact zero in the supplied probability column. They are retained as recorded; this notebook does not establish whether these are substantive estimates or mapping placeholders. Do not interpret them as proof of no automation risk.


In [ ]:
from pathlib import Path
from textwrap import fill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Standalone: this cell can run without executing the model simulation.
root = Path.cwd()
if not (root / "data" / "ipums_variables.csv").exists():
    root = root.parent
ranking = pd.read_csv(root / "data" / "ipums_variables.csv")
ranking = ranking[["id", "label", "auto_prob_average"]].copy()
assert ranking["id"].is_unique
assert ranking["auto_prob_average"].notna().all()
assert ranking["auto_prob_average"].between(0, 1).all()
assert len(ranking) >= 60
highest = ranking.sort_values(["auto_prob_average", "label"], ascending=[False, True]).head(30)
lowest = ranking.sort_values(["auto_prob_average", "label"], ascending=[True, True]).head(30)
assert len(highest) == len(lowest) == 30
assert set(highest.id).isdisjoint(lowest.id)

# Separate figures leave room for the long occupation names.
output = root / "results" / "fig"
output.mkdir(parents=True, exist_ok=True)
for frame, title, color, filename in [
    (highest, "30 occupations with the highest automation probability", "#cf4446", "automation_highest_30"),
    (lowest, "30 occupations with the lowest automation probability", "#2878ad", "automation_lowest_30"),
]:
    fig, ax = plt.subplots(figsize=(13, 19), layout="constrained")
    y = np.arange(len(frame))
    values = frame["auto_prob_average"].to_numpy()
    bars = ax.barh(y, values, color=color, height=0.7)
    ax.set_yticks(y, [fill(label, width=57) for label in frame["label"]], fontsize=9)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.set_xlim(0, 1.12 if filename == "automation_highest_30" else max(values) * 1.25)
    ax.bar_label(bars, labels=[f"{v:.2%}" for v in values], padding=4, fontsize=9)
    ax.set_xlabel("Estimated probability of computerization (not share of jobs lost)", labelpad=12)
    ax.set_title(title, fontsize=16, pad=20)
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)
    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    fig.savefig(output / f"{filename}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

# Exact values and full names remain available for inspection or export.
display(highest.reset_index(drop=True).rename_axis("rank").assign(rank=lambda d: np.arange(1, 31)))
display(lowest.reset_index(drop=True).rename_axis("rank").assign(rank=lambda d: np.arange(1, 31)))


## 7. Frey–Osborne shock: occupation-level unemployment effects
Recreate the two panels using the repository's saved numerical results for the empirical occupational mobility network (OMN) and the complete network (kn). Occupation bubbles use log-scaled employment, with capped sizes to reduce overlap, and colour indicates median annual earnings in US dollars; red dots represent the complete-network comparison. Hover over either network to see occupation names, employment, median earnings, automation probability and the percentage change in rate. Click the legend to hide a network, or drag to zoom. All 464 occupations are included.

The vertical axis is the relative percentage change in the rate: `100 * (transition rate - baseline rate) / baseline rate`, not a percentage-point change. The original experiment compares the pre-shock steady-state window (steps 25–74) with the 30-year transition window beginning at step 100. These are modeled outcomes under the paper's Frey–Osborne scenario, not observed unemployment changes.

The selected results use the original filename parameter suffix for separation 0.016, vacancy creation 0.012, adjustment 0.16, tau 3, a 30-year shock and the full-population setting (dimfact0). Long-term unemployment corresponds to roughly 27 weeks in the model. Plot styling is recreated; point values come from the supplied result files.

Earnings come from the supplied `median_earnings` column in `ipums_variables.csv`, matching the original plotting script. These are the paper's historical US earnings figures, not current wages or simulated post-shock wages. Both panels share one linear earnings colour scale.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

root = Path.cwd()
if not (root / "data" / "ipums_variables.csv").exists():
    root = root.parent
suffix = "FO_automation_deltau160v120gamma160_tau3_shockduration30_dimfact0.csv"
folder = root / "results" / "csv"
omn = pd.read_csv(folder / ("u_ltu_perc_changeOMN" + suffix))
complete = pd.read_csv(folder / ("u_ltu_perc_changekn" + suffix))
meta = pd.read_csv(root / "data" / "ipums_variables.csv")
emp = pd.read_csv(root / "data" / "ipums_employment_2016.csv")
# Match by occupation identifier; cross-check the employment classification.
assert np.array_equal(meta["acs_occ_code"].to_numpy(), emp["ipums_code"].to_numpy())
assert np.array_equal(meta["label"].to_numpy(), emp["ipums_ordered_occ_titles"].to_numpy())
meta["employment"] = emp["IPUMS_CPS_av_monthly_employment_whole_period"].to_numpy()
plot_data = meta[["id", "label", "auto_prob_average", "employment", "median_earnings"]].merge(
    omn[["id", "label", "u_perc_change", "ltu_perc_change"]],
    on=["id", "label"], validate="one_to_one").merge(
    complete[["id", "label", "u_perc_change", "ltu_perc_change"]],
    on=["id", "label"], validate="one_to_one", suffixes=("_omn", "_complete"))
assert len(plot_data) == len(meta) == 464
columns = ["auto_prob_average", "employment", "u_perc_change_omn", "ltu_perc_change_omn",
           "u_perc_change_complete", "ltu_perc_change_complete"]
assert np.isfinite(plot_data[columns].to_numpy()).all()
assert (plot_data.employment > 0).all()
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

# Bubble AREA follows log(1 + employment); largest diameter is capped at 19 px.
# Both panels use the same sizes. Complete-network points stay small.
log_employment = np.log1p(plot_data.employment.to_numpy())
marker_area = 12 + 72 * (log_employment - log_employment.min()) / np.ptp(log_employment)
custom = plot_data[["label", "employment", "u_perc_change_omn", "ltu_perc_change_omn",
                    "u_perc_change_complete", "ltu_perc_change_complete", "median_earnings"]].to_numpy()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.11,
                    subplot_titles=["(a) Unemployment rate", "(b) Long-term unemployment rate"])
for row, metric, own_index, complete_index in [
    (1, "u_perc_change", 2, 4), (2, "ltu_perc_change", 3, 5)
]:
    for key, name, color, index in [
        ("omn", "Occupational mobility network", "#218c57", own_index),
        ("complete", "Complete network", "#d64045", complete_index),
    ]:
        marker = dict(color=color, opacity=0.65 if key == "omn" else 0.85,
                      size=marker_area if key == "omn" else 6,
                      sizemode="area", line=dict(width=0))
        if key == "omn":
            marker.update(sizeref=2 * marker_area.max() / 19**2, sizemin=3,
                          color=plot_data.median_earnings, coloraxis="coloraxis", opacity=0.8)
        fig.add_trace(go.Scatter(
            x=plot_data.auto_prob_average, y=plot_data[metric + "_" + key],
            mode="markers", name=name, legendgroup=key, showlegend=(row == 1),
            customdata=custom, marker=marker,
            hovertemplate=("<b>%{customdata[0]}</b><br>" + name +
                           "<br>Automation level: %{x:.1%}<br>Employment: %{customdata[1]:,.0f}" +
                           "<br>Median annual earnings (USD): $%{customdata[6]:,.0f}" +
                           "<br>Change in rate: %{y:.2f}%<extra></extra>")), row=row, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1, row=row, col=1)
    fig.update_yaxes(title_text="Change in rate (%)", row=row, col=1)
fig.update_xaxes(range=[-0.03, 1.03], tickformat=".0%")
fig.update_xaxes(title_text="Automation level (probability of computerization)", row=2, col=1)
fig.update_layout(title="Impact of the Frey–Osborne automation shock",
                  template="plotly_white", height=950, hovermode="closest",
                  coloraxis=dict(colorscale="Viridis", cmin=plot_data.median_earnings.min(),
                                 cmax=plot_data.median_earnings.max(),
                                 colorbar=dict(title="Median annual<br>earnings (USD)", tickprefix="$", tickformat=",.0f")),
                  legend=dict(orientation="h", y=1.07, x=0), margin=dict(t=135),
                  dragmode="zoom")
output = root / "results" / "interactive"
output.mkdir(parents=True, exist_ok=True)
html = fig.to_html(full_html=False, include_plotlyjs=True,
                   config={"displaylogo": False, "scrollZoom": True, "responsive": True})
(output / "frey_osborne_occupation_unemployment_interactive.html").write_text(
    fig.to_html(full_html=True, include_plotlyjs=True,
                config={"displaylogo": False, "scrollZoom": True, "responsive": True}), encoding="utf-8")
assert np.isfinite(plot_data.median_earnings).all() and (plot_data.median_earnings > 0).all()
assert len(fig.data) == 4 and all(len(trace.x) == 464 for trace in fig.data)
assert np.isfinite(marker_area).all() and (marker_area > 0).all()
assert np.argsort(log_employment).tolist() == np.argsort(marker_area).tolist()
display({"application/vnd.plotly.v1+json": __import__("json").loads(fig.to_json()), "text/html": html}, raw=True)
print("Interactive charts: hover for occupation details; click legend items to hide a network; drag to zoom.")


## 8. Figure 6: network effects versus median earnings
For each occupation, subtract the complete-network percentage change from the occupational-network percentage change. Positive differences mean the network makes the modeled outcome worse; negative differences mean it improves the outcome relative to the complete-network comparison.

Both panels use historical median annual earnings from the paper's data on the horizontal axis. Bubble size uses log employment to reduce overlap. Hover for the occupation name, earnings, employment, automation level, both underlying percentage changes, and their difference. A difference of 20 means 20 percentage points between the **relative percentage changes**, not a 20-percentage-point increase in the underlying unemployment rate.

Interactive HTML copies of Figures 5 and 6 are saved to `results/interactive/` with their scripts included so they work offline in a browser.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

root = Path.cwd()
if not (root / "data" / "ipums_variables.csv").exists():
    root = root.parent
suffix = "FO_automation_deltau160v120gamma160_tau3_shockduration30_dimfact0.csv"
folder = root / "results" / "csv"
omn = pd.read_csv(folder / ("u_ltu_perc_changeOMN" + suffix))
complete = pd.read_csv(folder / ("u_ltu_perc_changekn" + suffix))
meta = pd.read_csv(root / "data" / "ipums_variables.csv")
emp = pd.read_csv(root / "data" / "ipums_employment_2016.csv")
# Match by occupation identifier; cross-check the employment classification.
assert np.array_equal(meta["acs_occ_code"].to_numpy(), emp["ipums_code"].to_numpy())
assert np.array_equal(meta["label"].to_numpy(), emp["ipums_ordered_occ_titles"].to_numpy())
meta["employment"] = emp["IPUMS_CPS_av_monthly_employment_whole_period"].to_numpy()
plot_data = meta[["id", "label", "auto_prob_average", "employment", "median_earnings"]].merge(
    omn[["id", "label", "u_perc_change", "ltu_perc_change"]],
    on=["id", "label"], validate="one_to_one").merge(
    complete[["id", "label", "u_perc_change", "ltu_perc_change"]],
    on=["id", "label"], validate="one_to_one", suffixes=("_omn", "_complete"))
assert len(plot_data) == len(meta) == 464
columns = ["auto_prob_average", "employment", "u_perc_change_omn", "ltu_perc_change_omn",
           "u_perc_change_complete", "ltu_perc_change_complete"]
assert np.isfinite(plot_data[columns].to_numpy()).all()
assert (plot_data.employment > 0).all()
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import json

# Differences between RELATIVE percentage changes, not raw unemployment rates.
plot_data["network_effect_u"] = plot_data.u_perc_change_omn - plot_data.u_perc_change_complete
plot_data["network_effect_ltu"] = plot_data.ltu_perc_change_omn - plot_data.ltu_perc_change_complete
log_employment = np.log1p(plot_data.employment.to_numpy())
marker_area = 12 + 72 * (log_employment - log_employment.min()) / np.ptp(log_employment)
custom = plot_data[["label", "employment", "median_earnings", "auto_prob_average",
                    "u_perc_change_omn", "u_perc_change_complete",
                    "ltu_perc_change_omn", "ltu_perc_change_complete"]].to_numpy()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                    subplot_titles=["(a) Network effects on unemployment", "(b) Network effects on long-term unemployment"])
for row, metric, omn_index, complete_index in [(1, "network_effect_u", 4, 5), (2, "network_effect_ltu", 6, 7)]:
    fig.add_trace(go.Scatter(
        x=plot_data.median_earnings / 1000, y=plot_data[metric], mode="markers",
        name="Occupations", showlegend=False, customdata=custom,
        marker=dict(size=marker_area, sizemode="area", sizeref=2 * marker_area.max() / 19**2,
                    sizemin=3, color="#218c57", opacity=0.7, line=dict(width=0)),
        hovertemplate=("<b>%{customdata[0]}</b><br>Median annual earnings: $%{customdata[2]:,.0f}" +
                       "<br>Employment: %{customdata[1]:,.0f}<br>Automation level: %{customdata[3]:.1%}" +
                       f"<br>Occupational network change: %{{customdata[{omn_index}]:.2f}}%" +
                       f"<br>Complete network change: %{{customdata[{complete_index}]:.2f}}%" +
                       "<br>Network effect: %{y:.2f} pp of percentage change<extra></extra>")), row=row, col=1)
    fig.add_hline(y=0, line_color="gray", line_dash="dash", line_width=1, row=row, col=1)
    fig.update_yaxes(title_text="Difference in percentage changes (pp)", row=row, col=1)
fig.update_xaxes(title_text="Median annual earnings (USD thousands)", row=2, col=1)
fig.update_layout(title="Network effects at the occupation level",
                  template="plotly_white", height=950, hovermode="closest", dragmode="zoom",
                  margin=dict(t=100, l=105))
assert len(fig.data) == 2 and all(len(t.x) == 464 for t in fig.data)
assert np.isfinite(plot_data[["network_effect_u", "network_effect_ltu"]]).all().all()
assert np.allclose(plot_data.network_effect_u, plot_data.u_perc_change_omn - plot_data.u_perc_change_complete)
output = root / "results" / "interactive"
output.mkdir(parents=True, exist_ok=True)
config = {"displaylogo": False, "scrollZoom": True, "responsive": True}
fig.write_html(output / "figure_6_network_effects_wages.html", include_plotlyjs=True, config=config)
display({"application/vnd.plotly.v1+json": json.loads(fig.to_json()),
         "text/html": fig.to_html(full_html=False, include_plotlyjs=True, config=config)}, raw=True)
print("Figure 6: 464 occupations per panel. Positive values mean detrimental network effects.")


## 9. Potential automation traps
This exploratory ranking combines high automation exposure with weak historical connections to lower-exposure occupations. It is an additional analysis, not a result claimed in the paper.

For row-normalized transition weights A and automation probabilities p:
- Access to safer destinations: `sum(A[i,j] * (1 - p[j]) for j != i)`.
- Trap score: `p[i] * (1 - safer_access[i])`. Higher means more exposed with weaker safer access.
- Exit weight: `sum(A[i,j] for j != i)`. Conditional safer access divides safer access by exit weight, separating the tendency to stay from the risk of destinations reached when leaving.

The chart's upper-left area is of interest. Colour shows the simulated long-term-unemployment network effect; red means the occupational network makes the change worse than the complete network. Bubble size uses log employment. The ranking is based on the trap score; the LTU effect is separate supporting evidence, not part of that score. The exported table also flags exposure >=70% with a positive LTU network effect; this is a descriptive filter, not a validated cutoff.

Limitations: historical flows are proxies, not transition barriers or current hiring probabilities; this score does not incorporate current vacancies, wages, geography, or retraining. The baseline network uses common self-loop weights, so exit weights may not differentiate occupations; inspect the printed range. The supplied automation column has 17 zeros of uncertain substantive/mapping status, which can inflate apparent access to safer destinations. Scores use those values as supplied and should be checked before policy use.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

root = Path.cwd()
if not (root / "data" / "ipums_variables.csv").exists():
    root = root.parent
suffix = "FO_automation_deltau160v120gamma160_tau3_shockduration30_dimfact0.csv"
folder = root / "results" / "csv"
omn = pd.read_csv(folder / ("u_ltu_perc_changeOMN" + suffix))
complete = pd.read_csv(folder / ("u_ltu_perc_changekn" + suffix))
meta = pd.read_csv(root / "data" / "ipums_variables.csv")
emp = pd.read_csv(root / "data" / "ipums_employment_2016.csv")
# Match by occupation identifier; cross-check the employment classification.
assert np.array_equal(meta["acs_occ_code"].to_numpy(), emp["ipums_code"].to_numpy())
assert np.array_equal(meta["label"].to_numpy(), emp["ipums_ordered_occ_titles"].to_numpy())
meta["employment"] = emp["IPUMS_CPS_av_monthly_employment_whole_period"].to_numpy()
plot_data = meta[["id", "label", "auto_prob_average", "employment", "median_earnings"]].merge(
    omn[["id", "label", "u_perc_change", "ltu_perc_change"]],
    on=["id", "label"], validate="one_to_one").merge(
    complete[["id", "label", "u_perc_change", "ltu_perc_change"]],
    on=["id", "label"], validate="one_to_one", suffixes=("_omn", "_complete"))
assert len(plot_data) == len(meta) == 464
columns = ["auto_prob_average", "employment", "u_perc_change_omn", "ltu_perc_change_omn",
           "u_perc_change_complete", "ltu_perc_change_complete"]
assert np.isfinite(plot_data[columns].to_numpy()).all()
assert (plot_data.employment > 0).all()
import plotly.graph_objects as go
from IPython.display import display
import json

# Normalize rows explicitly: scores describe historical transition weights,
# not observed applications, transition times, or guaranteed hiring chances.
raw = np.loadtxt(root / "data/occupational_mobility_network.csv", delimiter=",")
ids = plot_data.id.to_numpy(dtype=int)
assert raw.shape == (464, 464) and np.isfinite(raw).all() and (raw >= 0).all()
assert (raw.sum(axis=1) > 0).all()
A = raw / raw.sum(axis=1, keepdims=True)
assert np.allclose(A.sum(axis=1), 1)
# Work in id order, then attach scores by id rather than relying on merge order.
p_by_id = meta.set_index("id").loc[np.arange(len(raw)), "auto_prob_average"].to_numpy()
B = A.copy()
np.fill_diagonal(B, 0)
exit_weight = B.sum(axis=1)
safer_access = B @ (1 - p_by_id)
low_risk_access = B[:, p_by_id < 0.5].sum(axis=1)
conditional_safe = np.divide(safer_access, exit_weight, out=np.zeros_like(exit_weight), where=exit_weight > 0)
scores = pd.DataFrame({"id": np.arange(len(raw)), "exit_weight": exit_weight,
    "safer_access": safer_access, "low_risk_access": low_risk_access,
    "safer_access_conditional_on_exit": conditional_safe,
    "trap_score": p_by_id * (1 - safer_access)})
traps = plot_data.merge(scores, on="id", validate="one_to_one")
traps["ltu_network_effect"] = traps.ltu_perc_change_omn - traps.ltu_perc_change_complete
assert len(traps) == 464
assert traps[["trap_score", "safer_access", "exit_weight"]].apply(lambda col: col.between(0, 1).all()).all()
ranked = traps.sort_values(["trap_score", "label"], ascending=[False, True]).reset_index(drop=True)
ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))
ranked["high_exposure_and_detrimental_ltu_effect"] = (ranked.auto_prob_average >= 0.7) & (ranked.ltu_network_effect > 0)

log_emp = np.log1p(traps.employment.to_numpy())
area = 12 + 72 * (log_emp - log_emp.min()) / np.ptp(log_emp)
custom = traps[["label", "employment", "median_earnings", "trap_score", "exit_weight",
                "safer_access_conditional_on_exit", "ltu_network_effect", "low_risk_access"]].to_numpy()
fig = go.Figure(go.Scatter(
    x=traps.safer_access, y=traps.auto_prob_average, mode="markers",
    customdata=custom,
    marker=dict(size=area, sizemode="area", sizeref=2 * area.max() / 19**2, sizemin=3,
                color=traps.ltu_network_effect, colorscale="RdBu_r", cmid=0,
                colorbar=dict(title="LTU network<br>effect (pp)"), opacity=0.8),
    hovertemplate=("<b>%{customdata[0]}</b><br>Automation exposure: %{y:.1%}" +
        "<br>Access to safer destinations (weighted score): %{x:.3f}" +
        "<br>Trap score: %{customdata[3]:.3f}<br>Weight of leaving occupation: %{customdata[4]:.1%}" +
        "<br>Safer access conditional on leaving: %{customdata[5]:.3f}" +
        "<br>Weight of moves to occupations below 50% exposure: %{customdata[7]:.1%}" +
        "<br>LTU network effect: %{customdata[6]:.2f} pp" +
        "<br>Median earnings: $%{customdata[2]:,.0f}<br>Employment: %{customdata[1]:,.0f}<extra></extra>")))
fig.update_layout(title="Potential automation traps: high exposure and weak access to safer work",
    template="plotly_white", height=750, hovermode="closest",
    xaxis_title="Access to safer occupations (weighted score; higher is better)",
    yaxis_title="Automation probability", yaxis_tickformat=".0%", yaxis_range=[-0.03, 1.03])
fig.add_annotation(x=0.01, y=1, xref="paper", yref="paper", text="Potential traps: upper left",
                   showarrow=False, xanchor="left", yanchor="bottom")
assert len(fig.data[0].x) == 464
output = root / "results" / "interactive"
output.mkdir(parents=True, exist_ok=True)
config = {"displaylogo": False, "scrollZoom": True, "responsive": True}
fig.write_html(output / "potential_automation_traps.html", include_plotlyjs=True, config=config)
display({"application/vnd.plotly.v1+json": json.loads(fig.to_json()),
         "text/html": fig.to_html(full_html=False, include_plotlyjs=True, config=config)}, raw=True)
columns = ["rank", "label", "auto_prob_average", "safer_access", "trap_score", "exit_weight",
           "safer_access_conditional_on_exit", "ltu_network_effect", "median_earnings", "employment"]
display(ranked[columns].head(30))
ranked.to_csv(root / "results" / "csv" / "potential_automation_traps.csv", index=False)
print("Ranked all 464 occupations; showing the top 30. Scores are exploratory, not calibrated predictions.")
print("Exit-weight range:", round(exit_weight.min(), 4), "to", round(exit_weight.max(), 4))
